# Results and Findings

**ISB AMPBA · Speech Emotion Recognition on RAVDESS**

---

This notebook presents the **final results**. For how the project is built and why
each decision was made, start with `00_project_walkthrough.ipynb`.

As with that notebook, **no logic lives here** — every function comes from `ser/`
and is covered by tests. This notebook reads `results/results.csv` and narrates.

## The one-paragraph summary

We predict 8 emotions from ~3 seconds of speech. Held out by **speaker** — which
is what deployment looks like, since every real caller is someone the system has
never heard — we score **63.9% accuracy and 63.3% macro-F1**, against a 12.5%
chance rate.

Evaluated the way most published RAVDESS work does it, with a **random** split,
the same model scores **74.9%**. That 11-point difference is not a detail. We
proved its cause: our features can identify **which of 24 actors is speaking with
92.7% accuracy**.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

results = pd.read_csv(REPO / "results" / "results.csv")
results["accuracy"] = results["accuracy"].astype(float)
results["macro_f1"] = results["macro_f1"].astype(float)

print(f"{len(results)} logged runs · {results['model'].nunique()} models · "
      f"{results['feature_config'].nunique()} feature configurations")

173 logged runs · 9 models · 23 feature configurations


---
# 1. How we got from 49.8% to 63.3%

Every number below is pooled out-of-fold over all 1,440 clips, under the
**speaker-independent** protocol, on the frozen folds in `splits/folds.csv`.

The progression matters as much as the endpoint — it shows *what actually bought
the improvement*.

In [2]:
FLOOR = "mfcc40-d-dd-mean_std-trim"
WINNER = ("mfcc20-d-dd-mean_std-trim-chroma+spectral_centroid+"
          "spectral_contrast+zcr-per_speaker")

pooled = results[results["fold"] == "pooled"]
si = pooled[pooled["protocol"] == "speaker_independent"]

progression = pd.DataFrame([
    {"stage": "1. Untuned floor (40 MFCCs, no normalisation)",
     "macro_f1": si[(si.feature_config == FLOOR) & (si.model == "svm-rbf")].macro_f1.max()},
    {"stage": "2. + per-speaker normalisation, 20 MFCCs, extra descriptors",
     "macro_f1": si[(si.feature_config == WINNER) & (si.model == "svm-rbf")].macro_f1.max()},
    {"stage": "3. + hyperparameter tuning",
     "macro_f1": si[si.model == "svm-rbf-tuned"].macro_f1.max()},
])
progression["gain_pp"] = (progression["macro_f1"].diff() * 100).round(2)
progression.round(4)

,stage,macro_f1,gain_pp
0,"1. Untuned floor (40 MFCCs, no normalisation)",0.4980,NaN
1,"2. + per-speaker normalisation, 20 MFCCs, extra descriptors",0.6311,13.31
2,3. + hyperparameter tuning,0.6333,0.22


### Read the `gain_pp` column

**Feature design bought +13.31 points. Hyperparameter tuning bought +0.22.**

That is a ratio of about **60 to 1**, and it is the most practically useful thing
we learned. The fold-to-fold standard deviation is 5.09 points, so the tuning
gain is roughly one twenty-third of the noise — it is not a real improvement at
all.

If you take one methodological lesson from this project: on a dataset this size,
spend your time on *what the model sees*, not on *how it draws its boundary*.

---
# 2. The leakage gap — our headline finding

The same model, the same features, the same code. Only the **split** changes.

In [3]:
gap = (
    pooled[pooled.model.isin(["svm-rbf", "svm-rbf-tuned"])
           & (pooled.feature_config == WINNER)]
    .pivot_table(index="model", columns="protocol", values="accuracy", aggfunc="max")
)
gap["leakage_gap_pp"] = ((gap.random_stratified - gap.speaker_independent) * 100).round(2)
gap["inflation_%"] = ((gap.random_stratified / gap.speaker_independent - 1) * 100).round(1)
gap.round(4)

protocol,random_stratified,speaker_independent,statement_holdout,statement_holdout_si,leakage_gap_pp,inflation_%
model,,,,,,
svm-rbf,0.7271,0.6389,0.6194,0.5431,8.82,13.8
svm-rbf-tuned,0.7486,0.6389,NaN,NaN,10.97,17.2


### Two things to notice

**First:** a random split makes this model look like it scores **74.9%** when it
actually generalises at **63.9%**. Anyone reporting the former has not lied —
they have just measured something that does not correspond to deployment.

**Second, and less obvious: tuning made the leakage *worse*** — the gap widened
from 8.82 to 10.97 points. Tuning improved the random-split score by 2.15 points
and left the speaker-independent score completely untouched.

That is not coincidence. Under a random split, the inner cross-validation can
select hyperparameters that exploit speaker-specific structure, because the same
actors appear on both sides. Under the speaker-independent split there is no such
structure to find.

**Tuning against a leaky protocol manufactures improvement that does not exist.**
A tuned random-split RAVDESS number is inflated twice over.

---
# 3. Why the gap exists — we measured the mechanism

Rather than assert that the features "encode speaker identity", we tested it
directly: train a classifier to predict **which of the 24 actors** is speaking.

Chance is 1/24 = 4.2%.

In [4]:
leakage = pd.read_parquet(REPO / "features" / "phase7_leakage.parquet")
leakage.round(4)

,features,actor_id_accuracy,sd,chance,x_chance
0,raw,0.9271,0.0187,0.0417,22.3
1,per-speaker normalised,0.2118,0.0172,0.0417,5.1


**92.7% accuracy at identifying the speaker.** Twenty-two times chance.

Our features are, to a first approximation, a **speaker-recognition system that
we are asking to do emotion recognition**. Every contaminated result in the
literature follows from this.

Per-speaker normalisation cuts identifiability to 21.2% — which is exactly why it
was the single most valuable feature decision (+6.53 points). But 21.2% is still
5.1× chance: **residual identity survives**, and that is why the leakage gap
narrowed rather than closing.

---
# 4. Model comparison

Seven classical models, identical features, both protocols.

In [5]:
comparison = pd.read_parquet(REPO / "features" / "phase5_comparison.parquet")
si_models = comparison[comparison.protocol == "speaker_independent"].set_index("model")
rs_models = comparison[comparison.protocol == "random_stratified"].set_index("model")

table = pd.DataFrame({
    "SI_accuracy": si_models.accuracy,
    "SI_macro_f1": si_models.macro_f1,
    "SI_fold_sd": si_models.fold_sd,
    "RS_accuracy": rs_models.accuracy,
    "leakage_gap_pp": (rs_models.accuracy - si_models.accuracy) * 100,
}).sort_values("SI_macro_f1", ascending=False)
table.round(4)

,SI_accuracy,SI_macro_f1,SI_fold_sd,RS_accuracy,leakage_gap_pp
model,,,,,
svm-rbf,0.6389,0.6311,0.0509,0.7271,8.8194
xgboost,0.6042,0.5956,0.0620,0.6882,8.4028
logistic-regression,0.5931,0.5925,0.0498,0.6993,10.6250
random-forest,0.5688,0.5488,0.0665,0.6486,7.9861
gaussian-nb,0.5160,0.5071,0.0433,0.5486,3.2639
knn,0.4861,0.4602,0.0312,0.5764,9.0278
adaboost,0.4507,0.4504,0.0785,0.4757,2.5000


**SVM-RBF wins by 3.6 points of macro-F1 over XGBoost.**

Margin-based methods beat tree ensembles here, and the data shape explains why:
162 dense, continuous, standardised features over only 1,140 training clips.
Trees split one feature at a time and need volume to find good thresholds; a
kernel method draws one smooth boundary through the whole space.

**Always read the `SI_fold_sd` column.** With 24 actors in 5 folds, *which*
speakers land in the test fold matters a great deal — standard deviations run
from 3.1 to 7.9 points. A speaker-independent number quoted without its spread
is overstating what 5 folds of 24 actors can tell you.

---
# 5. Where the model fails

In [6]:
from ser.evaluate import evaluate_pooled
from ser.features import FeatureConfig, extract, feature_columns, per_speaker_normalise
from ser.metadata import load
from ser.models.classical import build
from ser.splits import get_split, load_folds
from ser import set_seeds

set_seeds()
config = FeatureConfig(normalisation="per_speaker", n_mfcc=20,
                       extras=("chroma", "spectral_contrast", "spectral_centroid", "zcr"))
meta = load()
frame = extract(config, meta)
cols = feature_columns(frame, meta)
indexed = frame.set_index("filename")
X = pd.DataFrame(
    per_speaker_normalise(indexed[cols].to_numpy(), indexed["actor"].to_numpy()),
    index=indexed.index, columns=cols,
)
y = indexed["emotion_label"]
folds = load_folds()

per_fold = []
for fold in range(5):
    train_files, test_files = get_split("speaker_independent", fold, folds=folds)
    model = build("svm-rbf")
    model.fit(X.loc[train_files].to_numpy(), y.loc[train_files].to_numpy())
    per_fold.append((y.loc[test_files].to_numpy(),
                     model.predict(X.loc[test_files].to_numpy()),
                     indexed.loc[test_files].reset_index()))

# append=False: reproducing a logged result, not creating a new experiment.
best = evaluate_pooled(per_fold, "speaker_independent", model="svm-rbf",
                       feature_config=config.name, notes="notebook reproduction",
                       append=False)
print(f"accuracy {best['accuracy']:.4f}   macro-F1 {best['macro_f1']:.4f}   n={best['n']}")

accuracy 0.6389   macro-F1 0.6311   n=1440


In [7]:
best["confusion_matrix"].style.background_gradient(cmap="Blues", axis=None).format("{:.0f}")

predicted,neutral,calm,happy,sad,angry,fearful,disgust,surprised
true,,,,,,,,
neutral,47,13,10,22,0,2,0,2
calm,8,143,2,32,0,3,4,0
happy,9,3,130,6,8,26,3,7
sad,7,43,23,80,2,16,17,4
angry,3,0,10,2,134,15,18,10
fearful,1,11,24,21,7,105,10,13
disgust,1,3,2,8,13,9,148,8
surprised,2,1,16,5,11,18,6,133


In [8]:
best["per_class"].sort_values("f1", ascending=False).round(3)

,emotion,precision,recall,f1,support
6,disgust,0.718,0.771,0.744,192
4,angry,0.766,0.698,0.730,192
7,surprised,0.751,0.693,0.721,192
1,calm,0.659,0.745,0.699,192
2,happy,0.599,0.677,0.636,192
5,fearful,0.541,0.547,0.544,192
0,neutral,0.603,0.490,0.540,96
3,sad,0.455,0.417,0.435,192


**Sad (F1 0.435) and neutral (F1 0.540) are the weak classes**, exactly as the
Phase 1 audit predicted: neutral has half the training data and the shortest
speech after trimming.

The three dominant confusions are **sad→calm**, **happy→fearful** and
**neutral→calm**. None of them is random — the first and third are low-arousal
classes collapsing together, the second is a high-arousal pair.

In [9]:
structure = pd.read_parquet(REPO / "features" / "phase7_structure.parquet")
structure.round(4)

,within_arousal_errors,across_arousal_errors,within_share,chance_within_share,lift
0,323,197,0.6212,0.4286,1.4494


**62.1% of all errors stay inside an arousal band, against 42.9% expected by
chance — a 1.45× lift.**

The model reliably works out *how activated* the speaker is, and then struggles
to say which emotion at that activation level. That is the single most compact
description of what it has and has not learned.

---
# 6. Fairness and deployment slices

A single headline number hides who the model fails.

In [10]:
print("=== BY GENDER ===")
display(best["slices"]["gender"].round(4))
print("=== BY INTENSITY ===")
display(best["slices"]["intensity_label"].round(4))

=== BY GENDER ===


,level,n,accuracy,macro_f1
0,female,720,0.6958,0.6923
1,male,720,0.5819,0.5646


=== BY INTENSITY ===


,level,n,accuracy,macro_f1
0,normal,768,0.5990,0.6003
1,strong,672,0.6845,0.5984


### Gender: a 12.8-point macro-F1 gap, worse on male speakers

The dataset is **perfectly balanced** by gender — 12 actors and 720 clips each —
so this is not a data-quantity effect. The likely mechanism is in the Phase 2
prosody analysis: female F0 averages 288 Hz against 178 Hz for male, a 1.62×
difference *larger than any between-emotion difference*. The emotional pitch
excursions that carry the signal are compressed into a narrower band for male
speakers.

**This has to be disclosed.** A contact-centre QA system 12.8 points worse at
reading male callers is a deployment risk with a fairness dimension.

### Intensity: the obvious reading is wrong

Accuracy is 8.6 points higher on strong-intensity clips, which looks like clean
confirmation that acted-up emotion is easier. But **macro-F1 is essentially
identical** (0.6003 vs 0.5984).

The reason: **the strong subset contains no neutral clips at all**, because
RAVDESS never recorded strong-intensity neutral. So `strong` is a 7-class problem
that excludes our hardest class, while `normal` is the full 8-class problem. Part
of the apparent advantage is an easier label set, not easier audio.

Macro-F1 weights every class equally and sees through this. It is a good example
of why we report both metrics on every single run.

---
# 7. What we tried that did not work

Reported in the same detail as the successes. Full write-up:
`docs/novelty_findings.md`.

In [11]:
two_stage = pd.read_parquet(REPO / "features" / "phase7_two_stage.parquet")
lexical = pd.read_parquet(REPO / "features" / "phase7_lexical.parquet")
display(two_stage.round(4))
display(lexical.round(4))

,model,accuracy,macro_f1,stage1_arousal_accuracy
0,two-stage (arousal first),0.6194,0.6149,0.8507


,protocol,accuracy,macro_f1,n
0,statement_holdout,0.6194,0.6053,720
1,statement_holdout_si,0.5431,0.5175,720
2,speaker_independent,0.6389,0.6311,1440


### The arousal-first two-stage model failed (−1.62 points)

**Hypothesis:** since errors cluster by arousal, predict high/low arousal first,
then emotion within that branch. Each specialist gets an easier problem.

**Why it lost:** the arousal gate is only **85.1%** accurate, and its mistakes are
**unrecoverable**. A clip routed to the wrong specialist cannot receive the right
label — that label is not in the specialist's output set. Roughly 15% of clips
are condemned before the second stage begins.

The flat model can hedge across the arousal boundary; hard routing throws that
away, and on this data it costs more than specialisation gains. **The diagnosis
was correct and the remedy still failed.** Soft routing — weighting each
specialist by the gate's probability rather than committing — is the natural
follow-up and is not yet tested.

### Lexical invariance: good news

Changing the **sentence** costs only −2.6 points. Changing sentence **and**
speaker costs −11.4. The 8.8-point difference is the part attributable to
*speaker familiarity* rather than lexical generalisation.

**Prosody generalises across content almost for free** — the model is reading
delivery, not memorising the acoustics of particular words.

---
# 8. Honest limitations

| Limitation | Why it matters |
|---|---|
| **RAVDESS is acted, scripted, studio-recorded** | Spontaneous emotion is subtler. Expect materially below 63.9% on real calls. |
| **24 actors is few** | Fold sd 5.09 pp. The 95% interval on our headline is roughly ±7.5 pp. |
| **Per-speaker normalisation is transductive** | It uses held-out audio (no labels). Standard CMVN practice, but a strict reviewer may discount it. |
| **Neutral remains weak** (F1 0.540) | Real audio is mostly neutral, so this is the binding constraint on deployment. |
| **Deep-learning tracks not included here** | Phase 6 is owned separately; it must beat **0.6333** macro-F1 to justify its complexity. |

## Reproducing everything

```bash
.venv/Scripts/python.exe -m pytest -q        # 163 tests
.venv/Scripts/python.exe -m ser.run_ablation # Phase 4
.venv/Scripts/python.exe -m ser.run_phase5   # Phase 5
.venv/Scripts/python.exe -m ser.run_phase7   # Phase 7
```

Every run appends to `results/results.csv` under the folds frozen in
`splits/folds.csv`. Those assignments have not changed since 2026-08-22 and a
test fails if they ever do.